# Methods versus number of samples (for finite case)

## Set-up

In [1]:
from utils_3 import PairwiseData, Population
from datasets import load_dataset

ds = load_dataset("lmarena-ai/arena-human-preference-140k")
ds = dict(ds)

pw = PairwiseData(ds, M=30, N=30)
population = Population(pw)

/home/jennifer/miniconda3/envs/ld/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 30/30 [00:00<00:00, 30.02it/s]


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit
from scipy.optimize import linprog
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

## Set-up utilities

In [3]:
def ij_from_pairwise(w_arr, l_arr, n_items):
    ij_wins = np.zeros((n_items, n_items))
    np.add.at(ij_wins, (w_arr, l_arr), 1.0)
    return ij_wins

In [4]:
def _solve_maximal_lottery(margins, tie_tol=1e-12):
    C = margins.shape[0]
    result = linprog(
        c=np.zeros(C),
        A_ub=-margins.T,
        b_ub=np.zeros(C),
        A_eq=np.ones((1, C)),
        b_eq=np.array([1.0]),
        bounds=[(0.0, 1.0)] * C,
        method='highs',
    )
    lottery = np.maximum(result.x if result.success else np.ones(C) / C, 0)
    lottery[lottery < tie_tol] = 0.0
    s = lottery.sum()
    return lottery / s if s > 0 else np.ones(C) / C

def borda_ranking(ij_wins, with_scores=False):
    num = ij_wins.sum(axis=1)
    denom = ij_wins.sum(axis=1) + ij_wins.sum(axis=0)
    denom = np.where(denom == 0.0, 1.0, denom)
    scores = num / denom
    if with_scores:
        return np.argsort(-scores), scores
    else:
        return np.argsort(-scores)

def borda_peeling_ranking(ij_wins):
    remaining = list(range(ij_wins.shape[0]))
    ranking = []
    while remaining:
        rem = np.array(remaining)
        ij_remaining = ij_wins[np.ix_(rem, rem)]
        _, scores = borda_ranking(ij_remaining, with_scores=True)
        w = int(np.argmax(scores))
        ranking.append(remaining[w])
        remaining.pop(w)
    return np.array(ranking)

def copeland_ranking(ij_wins, tol=1e-9):
    denom = (ij_wins + ij_wins.T)
    denom = np.where(denom == 0.0, 1.0, denom)
    m = (ij_wins - ij_wins.T) / denom
    return np.argsort(-((m > tol).sum(axis=1) - (m < -tol).sum(axis=1)))

def copeland_peeling_ranking(ij_wins, tol=1e-9):
    remaining = list(range(ij_wins.shape[0]))
    ranking = []
    while remaining:
        rem = np.array(remaining)
        ij_remaining = ij_wins[np.ix_(rem, rem)]
        denom = ij_remaining + ij_remaining.T
        m = (ij_remaining - ij_remaining.T) / np.where(denom == 0.0, 1.0, denom)
        scores = (m > tol).sum(axis=1) - (m < -tol).sum(axis=1)
        w = int(np.argmax(scores))
        ranking.append(remaining[w]); remaining.pop(w)
    return np.array(ranking)

def ml_argmax_ranking(ij_wins, tol=1e-12):
    remaining = list(range(ij_wins.shape[0]))
    ranking = []
    while remaining:
        rem = np.array(remaining)
        sub = ij_wins[np.ix_(rem, rem)]
        with np.errstate(invalid='ignore', divide='ignore'):
            margins = np.where(sub + sub.T > 0, (sub - sub.T) / (sub + sub.T), 0.0)
        lot = _solve_maximal_lottery(margins, tol)
        w = int(np.argmax(lot))
        ranking.append(remaining[w]); remaining.pop(w)
    return np.array(ranking)

def ml_nonzero_ranking(ij_wins, tol=1e-12):
    remaining = list(range(ij_wins.shape[0]))
    ranking = []
    while remaining:
        rem = np.array(remaining)
        sub = ij_wins[np.ix_(rem, rem)]
        with np.errstate(invalid='ignore', divide='ignore'):
            margins = np.where(sub + sub.T > 0, (sub - sub.T) / (sub + sub.T), 0.0)
        lot = _solve_maximal_lottery(margins, tol)
        order = np.argsort(-lot)
        nonzero = order[lot[order] > tol]
        if len(nonzero) == 0:
            nonzero = np.array([int(np.argmax(lot))])
        selected = [remaining[i] for i in nonzero]
        ranking.extend(selected)
        remaining = [c for c in remaining if c not in set(selected)]
    return np.array(ranking)

In [5]:
def leaderboard_dist_w(ranking, true_ranking, avg_utils, w):
    ranking = np.asarray(ranking)
    true_ranking = np.asarray(true_ranking)

    ranking_utils = avg_utils[ranking]
    true_ranking_utils = avg_utils[true_ranking]

    num = true_ranking_utils * w
    denom = ranking_utils * w

    denom_sum = denom.sum()
    ratio = num.sum() / denom_sum if denom_sum > 0 else np.inf

    return ratio, None

def leaderboard_dist(ranking, true_ranking, avg_utils):
    ranking = np.asarray(ranking)
    true_ranking = np.asarray(true_ranking)

    ranking_utils = avg_utils[ranking]
    true_ranking_utils = avg_utils[true_ranking]

    denom_cumsum = np.cumsum(ranking_utils)
    num_cumsum = np.cumsum(true_ranking_utils)

    valid = denom_cumsum > 0
    ratios = np.where(valid, num_cumsum / np.where(denom_cumsum > 0, denom_cumsum, 1.0), -np.inf)
    k = int(np.argmax(ratios))
    ratio = ratios[k] if valid.any() else np.inf
    return ratio, k

In [6]:
def ranking_distribution_pruned(candidates, maximal_lottery_fn):
    """Returns (ranking_dist, cache). Cache maps frozenset -> {candidate: prob}."""
    cache = {}

    def get_lottery(S):
        S = frozenset(S)
        if S not in cache:
            cache[S] = maximal_lottery_fn(list(S))
        return cache[S]

    def recurse(remaining):
        remaining = frozenset(remaining)

        if len(remaining) == 0:
            return {(): 1.0}

        ml = get_lottery(remaining)

        invalid = set(ml) - remaining
        if invalid:
            raise ValueError(f"Lottery returned candidates not in remaining set: {invalid}")

        total_positive = False
        result = {}

        for c, p in ml.items():
            if p == 0:
                continue
            total_positive = True
            for suffix, q in recurse(remaining - {c}).items():
                result[(c,) + suffix] = p * q

        if not total_positive:
            raise ValueError(f"Lottery has no positive-probability candidates for set {remaining}")

        return result

    return recurse(frozenset(candidates)), cache
    

def argmax_from_cache(cache, candidates):
    """Derive ml_argmax ranking from cache — no LP solves needed."""
    remaining = frozenset(candidates)
    ranking = []
    while len(remaining) > 1:
        lottery = cache[remaining]
        winner = max(lottery, key=lambda c: lottery[c])
        ranking.append(winner)
        remaining = remaining - {winner}
    ranking.append(next(iter(remaining)))
    return np.array(ranking)


def nonzero_from_cache(cache, candidates, tol=1e-12):
    """Derive ml_nonzero ranking from cache — no LP solves needed.

    ranking_distribution_pruned recurses through all orderings of nonzero
    candidates, so every subset reachable by removing nonzero candidates
    one-by-one is guaranteed to be in the cache.
    """
    remaining = frozenset(candidates)
    ranking = []
    while len(remaining) > 0:
        if len(remaining) == 1:
            ranking.append(next(iter(remaining)))
            break
        lottery = cache[remaining]
        nonzero = sorted(
            [c for c, p in lottery.items() if p > tol],
            key=lambda c: -lottery[c],
        )
        if not nonzero:
            nonzero = [max(lottery, key=lambda c: lottery[c])]
        ranking.extend(nonzero)
        remaining = remaining - frozenset(nonzero)
    return np.array(ranking)


def expected_average_utilities(ranking_dist, avg_utilities):
    rankings = np.array(list(ranking_dist.keys()), dtype=int)  # (K, M)
    probs    = np.array(list(ranking_dist.values()))            # (K,)
    return (probs[:, None] * avg_utilities[rankings]).sum(axis=0)


def expected_leaderboard_distortion(ranking_dist, true_ranking, avg_utils):
    true_ranking = np.asarray(true_ranking)
    ranking_utils = expected_average_utilities(ranking_dist, avg_utils)
    true_ranking_utils = avg_utils[true_ranking]
    denom_cumsum = np.cumsum(ranking_utils)
    num_cumsum   = np.cumsum(true_ranking_utils)
    valid = denom_cumsum > 0
    ratios = np.where(valid, num_cumsum / np.where(denom_cumsum > 0, denom_cumsum, 1.0), -np.inf)
    return float(np.max(ratios)) if valid.any() else np.inf

def expected_leaderboard_distortion_w(ranking_dist, true_ranking, avg_utils, w):
    # ranking = np.asarray(ranking)
    true_ranking = np.asarray(true_ranking)

    ranking_utils = expected_average_utilities(ranking_dist=ranking_dist, avg_utilities=avg_utils)
    true_ranking_utils = avg_utils[true_ranking]

    denom = (ranking_utils * w).sum()
    num = (true_ranking_utils * w).sum()

    ratio = num / denom if denom > 0 else np.inf
    return ratio


def make_ml_fn_from_ij_wins(ij_wins):
    def fn(candidates):
        sub = ij_wins[np.ix_(candidates, candidates)]
        with np.errstate(invalid='ignore', divide='ignore'):
            margins = np.where(sub + sub.T > 0, (sub - sub.T) / (sub + sub.T), 0.0)
        lot = _solve_maximal_lottery(margins)
        return {c: float(lot[i]) for i, c in enumerate(candidates)}
    return fn

In [7]:
def sampled_ranking_dist(candidates, ij_wins, tol=1e-12, rounds=10):
    lp_cache = {}  # frozenset(remaining) -> lottery array, shared across rounds
    dist = {}

    for _ in range(rounds):
        remaining = list(candidates)
        ranking = []
        while remaining:
            key = frozenset(remaining)
            if key not in lp_cache:
                rem = np.array(remaining)
                sub = ij_wins[np.ix_(rem, rem)]
                with np.errstate(invalid='ignore', divide='ignore'):
                    margins = np.where(sub + sub.T > 0, (sub - sub.T) / (sub + sub.T), 0.0)
                lp_cache[key] = _solve_maximal_lottery(margins, tol)
            lot = lp_cache[key]
            w = np.random.choice(len(remaining), p=lot)
            ranking.append(remaining[w]); remaining.pop(w)

        r = tuple(ranking)
        dist[r] = dist.get(r, 0.0) + 1.0 / rounds

    return dist

In [8]:
true_ranking = np.argsort(-population.avg_utilities)
candidates = np.arange(pw.M)

In [9]:
w = 1 / (1.1 ** np.arange(pw.M))

In [10]:
empirical_pair_distribution = np.zeros((pw.M, pw.M))

for i in range(len(pw.winners)):
    first = max(pw.winners[i], pw.losers[i])
    second = min(pw.winners[i], pw.losers[i])
    empirical_pair_distribution[first, second] += 1.0

empirical_pair_distribution /= empirical_pair_distribution.sum()

In [11]:
SMALL_BETAS = np.asarray([0.01, 0.1, 0.5, 1.0, 2.0, ])
BETAS = np.asarray([3.0, 5.0, 10.0, 15.0, 20, 25, 30.0, 35.0, 40.0, 45.0, 50.0])
betas = np.concat([SMALL_BETAS, BETAS])

In [12]:
np.random.seed(1001)

# 30_000 Samples

In [13]:
NUM_SAMPLES = 30_000
NUM_ROUNDS = 10

In [14]:
betas = np.concat([SMALL_BETAS, BETAS])

methods = ['borda', 'borda_peeling', 'copeland', 'copeland_peeling', 'ml_argmax', 'ml_nonzero', 'ml_sampling']
betas_distortions = {m: {float(b): [] for b in betas} for m in methods}
supremum_distortions = {m: {float(b): [] for b in betas} for m in methods}

# sample pairs once — only Bernoulli draws vary across rounds
pairs = np.array(np.meshgrid(np.arange(pw.M), np.arange(pw.M))).T.reshape(-1, 2)
flattened_empirical_pair_distribution = empirical_pair_distribution.ravel()

indices = np.random.choice(np.arange(len(flattened_empirical_pair_distribution)), size=NUM_SAMPLES, p=flattened_empirical_pair_distribution)
coords = np.array(np.unravel_index(indices, empirical_pair_distribution.shape)).T
model_As_fixed = coords[..., 0]
model_Bs_fixed = coords[..., 1]

u_diff = population.population_utilities[:, model_As_fixed] - population.population_utilities[:, model_Bs_fixed]

In [15]:
for beta in tqdm(betas):
    p = (expit(beta * u_diff) * population.voter_distr[:, None]).sum(axis=0)

    for _ in tqdm(range(NUM_ROUNDS)):
        mask = np.random.rand(NUM_SAMPLES) < p
        w_arr = np.where(mask, model_As_fixed, model_Bs_fixed)
        l_arr = np.where(mask, model_Bs_fixed, model_As_fixed)
        ij_wins = ij_from_pairwise(w_arr, l_arr, pw.M)

        for m, fn in [
            ('borda',            borda_ranking),
            ('borda_peeling',    borda_peeling_ranking),
            ('copeland',         copeland_ranking),
            ('copeland_peeling', copeland_peeling_ranking),
        ]:
            w_dist, _ = leaderboard_dist_w(fn(ij_wins), true_ranking, population.avg_utilities, w=w)
            dist, _ = leaderboard_dist(fn(ij_wins), true_ranking, population.avg_utilities)
            betas_distortions[m][float(beta)].append(w_dist)
            supremum_distortions[m][float(beta)].append(dist)

        for m, ranking in [
            ('ml_argmax',  ml_argmax_ranking(ij_wins)),
            ('ml_nonzero', ml_nonzero_ranking(ij_wins)),
        ]:
            w_dist, _ = leaderboard_dist_w(ranking, true_ranking, population.avg_utilities, w=w)
            dist, _ = leaderboard_dist(ranking, true_ranking, population.avg_utilities)
            betas_distortions[m][float(beta)].append(w_dist)
            supremum_distortions[m][float(beta)].append(dist)

        # ml_sampling
        ranking_dist_sampled = sampled_ranking_dist(list(range(pw.M)), ij_wins, rounds=1000)
        dist   = expected_leaderboard_distortion(ranking_dist_sampled, true_ranking, population.avg_utilities)
        w_dist = expected_leaderboard_distortion_w(ranking_dist_sampled, true_ranking, population.avg_utilities, w=w)
        betas_distortions['ml_sampling'][float(beta)].append(w_dist)
        supremum_distortions['ml_sampling'][float(beta)].append(dist)

  6%|▋         | 1/16 [09:44<2:26:02, 584.16s/it]


KeyboardInterrupt: 

In [ ]:
method_style = {
    'borda':            dict(color='C0', marker='o', linestyle='-',  label='Borda'),
    'borda_peeling':    dict(color='C1', marker='s', linestyle='--', label='Borda peeling'),
    'copeland':         dict(color='C2', marker='^', linestyle='-',  label='Copeland'),
    'copeland_peeling': dict(color='C3', marker='D', linestyle='--', label='Copeland peeling'),
    'ml_argmax':        dict(color='C4', marker='P', linestyle='-',  label='ML argmax'),
    'ml_nonzero':       dict(color='C5', marker='*', linestyle='--', label='ML nonzero'),
    'ml_sampling':      dict(color='C6', marker='h', linestyle='-',  label='ML sampling (expected)', linewidth=2),
}

fig, ax = plt.subplots(figsize=(10, 5))
for m, kw in method_style.items():
    means = np.array([np.mean(betas_distortions[m][float(b)]) for b in betas])
    stds  = np.array([np.std( betas_distortions[m][float(b)]) for b in betas])
    ax.plot(betas, means, **kw)
    ax.fill_between(betas, means - stds, means + stds, alpha=0.15, color=kw['color'])

# ax.set_xscale('log')
ax.set_xlabel('beta')
ax.set_ylabel('distortion')
ax.set_title(f'Distortion vs beta  (M={pw.M}, {NUM_SAMPLES} samples/round, {NUM_ROUNDS} rounds)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
method_style = {
    'borda':            dict(color='C0', marker='o', linestyle='-',  label='Borda'),
    'borda_peeling':    dict(color='C1', marker='s', linestyle='--', label='Borda peeling'),
    'copeland':         dict(color='C2', marker='^', linestyle='-',  label='Copeland'),
    'copeland_peeling': dict(color='C3', marker='D', linestyle='--', label='Copeland peeling'),
    'ml_argmax':        dict(color='C4', marker='P', linestyle='-',  label='ML argmax'),
    'ml_nonzero':       dict(color='C5', marker='*', linestyle='--', label='ML nonzero'),
    'ml_sampling':      dict(color='C6', marker='h', linestyle='-',  label='ML sampling (expected)', linewidth=2),
}

fig, ax = plt.subplots(figsize=(10, 5))
for m, kw in method_style.items():
    means = np.array([np.mean(supremum_distortions[m][float(b)]) for b in betas])
    stds  = np.array([np.std( supremum_distortions[m][float(b)]) for b in betas])
    ax.plot(betas, means, **kw)
    ax.fill_between(betas, means - stds, means + stds, alpha=0.15, color=kw['color'])

# ax.set_xscale('log')
ax.set_xlabel('beta')
ax.set_ylabel('distortion')
ax.set_title(f'Supremum Distortion vs beta  (M={pw.M}, {NUM_SAMPLES} samples/round, {NUM_ROUNDS} rounds)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()